# 18p — June 2026 CLOB market-price recovery

This notebook extends the verified 18l market-recovery method to the 30 certified June 2026 Hong Kong temperature event books.

It:

- requires the completed 18o June realised-outcome panel;
- retrieves official Polymarket CLOB YES-token price histories;
- uses the same seven-day history window and 60-minute fidelity as 18l;
- constructs four Hong Kong local-time decision cut-offs:
  - `24h_prior`: 00:00 HKT on the day before the event;
  - `12h_prior`: 12:00 HKT on the day before the event;
  - `6h_prior`: 18:00 HKT on the day before the event;
  - `event_day_open`: 00:00 HKT on the event date;
- selects only the latest recovered price at or before each cut-off;
- records missing prices rather than imputing or looking forward;
- calculates market-only Brier and logarithmic scores;
- constructs event-book probability-sum and categorical diagnostics;
- writes canonical raw-history, decision, scoring, issue, report and SHA-256 manifest outputs.

The official CLOB endpoint is:

`GET https://clob.polymarket.com/prices-history`

with query parameters `market=<YES token id>`, `startTs`, `endTs`, and `fidelity=60`.

The notebook never creates a branch, commit, push, pull request, reminder or notification.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
import requests
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / ".git").exists():
    raise RuntimeError(
        "Run this notebook from the repository root. "
        f"Current directory: {REPO_ROOT}"
    )

STEP = "18p"
HKT = ZoneInfo("Asia/Hong_Kong")
UTC = timezone.utc

OUTCOME_PATH = (
    REPO_ROOT
    / "data/processed/18o_june_2026_hko_realised_outcomes"
    / "18o_june_2026_contract_outcomes.csv"
)
OUTCOME_SUMMARY_PATH = (
    REPO_ROOT
    / "data/processed/18o_june_2026_hko_realised_outcomes"
    / "18o_june_2026_outcome_summary.json"
)

RAW_DIR = REPO_ROOT / "data/raw/18p_june_2026_clob_market_price_recovery"
OUT_DIR = REPO_ROOT / "data/processed/18p_june_2026_clob_market_price_recovery"
REPORT_DIR = REPO_ROOT / "reports/18p_june_2026_clob_market_price_recovery"

for directory in (RAW_DIR, OUT_DIR, REPORT_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CLOB_BASE_URL = "https://clob.polymarket.com"
PRICE_HISTORY_URL = f"{CLOB_BASE_URL}/prices-history"
USER_AGENT = (
    "2026MScWeatherForecastingPolymarket/"
    "18p-june-2026-clob-market-price-recovery"
)

HISTORY_WINDOW_DAYS = 7
HISTORY_FIDELITY_MINUTES = 60
REQUEST_TIMEOUT_SECONDS = 90
MAX_ATTEMPTS = 5
REQUEST_SLEEP_SECONDS = 0.05
LOG_EPSILON = 1e-6

DECISION_RULE_OFFSETS_HOURS = {
    "24h_prior": -24,
    "12h_prior": -12,
    "6h_prior": -6,
    "event_day_open": 0,
}
DECISION_RULE_ORDER = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]

session = requests.Session()
session.headers.update(
    {
        "User-Agent": USER_AGENT,
        "Accept": "application/json",
    }
)

if not OUTCOME_PATH.is_file():
    raise FileNotFoundError(
        "The completed 18o outcome panel is missing: "
        f"{OUTCOME_PATH}"
    )

print(f"Repository root: {REPO_ROOT}")
print(f"Outcome input: {OUTCOME_PATH.relative_to(REPO_ROOT)}")
print(f"History window: {HISTORY_WINDOW_DAYS} days")
print(f"History fidelity: {HISTORY_FIDELITY_MINUTES} minutes")

Repository root: /Users/edwardlee/Desktop/2026MScWeatherForecastingPolymarket
Outcome input: data/processed/18o_june_2026_hko_realised_outcomes/18o_june_2026_contract_outcomes.csv
History window: 7 days
History fidelity: 60 minutes


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def iso_utc_from_unix(seconds: int | float) -> pd.Timestamp:
    return pd.to_datetime(seconds, unit="s", utc=True)


def request_price_history(
    *,
    token_id: str,
    start_ts: int,
    end_ts: int,
) -> tuple[dict[str, Any], dict[str, Any]]:
    params = {
        "market": token_id,
        "startTs": start_ts,
        "endTs": end_ts,
        "fidelity": HISTORY_FIDELITY_MINUTES,
    }
    last_error = ""

    for attempt in range(1, MAX_ATTEMPTS + 1):
        requested_at = datetime.now(UTC)
        try:
            response = session.get(
                PRICE_HISTORY_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            metadata = {
                "selected_yes_token_id": token_id,
                "requested_at_utc": requested_at.isoformat(),
                "request_url": response.url,
                "start_ts": start_ts,
                "end_ts": end_ts,
                "fidelity_minutes": HISTORY_FIDELITY_MINUTES,
                "attempt": attempt,
                "status_code": response.status_code,
                "error": "",
            }

            if response.status_code == 429 or response.status_code >= 500:
                last_error = f"HTTP {response.status_code}"
                time.sleep(1.5 * attempt)
                continue

            response.raise_for_status()
            payload = response.json()

            if not isinstance(payload, dict):
                raise ValueError(
                    f"Expected JSON object, found {type(payload).__name__}"
                )

            history = payload.get("history", [])
            if history is None:
                history = []
            if not isinstance(history, list):
                raise ValueError("The response 'history' field is not a list")

            metadata.update(
                {
                    "has_history": len(history) > 0,
                    "history_points_returned": len(history),
                }
            )
            return payload, metadata

        except (requests.RequestException, ValueError) as exc:
            last_error = f"{type(exc).__name__}: {exc}"
            time.sleep(1.5 * attempt)

    raise RuntimeError(
        f"Price-history request failed for token {token_id}: {last_error}"
    )


def normalise_history_points(
    *,
    payload: dict[str, Any],
    token_id: str,
    event_date: pd.Timestamp,
    raw_path: Path,
) -> list[dict[str, Any]]:
    output: list[dict[str, Any]] = []

    for source_index, point in enumerate(payload.get("history", [])):
        if not isinstance(point, dict):
            continue

        try:
            timestamp_unix = int(float(point["t"]))
            price = float(point["p"])
        except (KeyError, TypeError, ValueError):
            continue

        if not math.isfinite(price):
            continue

        output.append(
            {
                "event_date": event_date.date().isoformat(),
                "selected_yes_token_id": token_id,
                "price_timestamp_unix": timestamp_unix,
                "price_timestamp_utc": iso_utc_from_unix(timestamp_unix),
                "price_timestamp_hkt": iso_utc_from_unix(timestamp_unix).tz_convert(HKT),
                "p_market": price,
                "source_index": source_index,
                "raw_response_path": str(raw_path.relative_to(REPO_ROOT)),
            }
        )

    return output


def decision_times(event_date: pd.Timestamp) -> list[dict[str, Any]]:
    event_open_hkt = pd.Timestamp(
        year=event_date.year,
        month=event_date.month,
        day=event_date.day,
        hour=0,
        minute=0,
        second=0,
        tz=HKT,
    )

    rows: list[dict[str, Any]] = []
    for rule in DECISION_RULE_ORDER:
        cutoff_hkt = event_open_hkt + pd.Timedelta(
            hours=DECISION_RULE_OFFSETS_HOURS[rule]
        )
        rows.append(
            {
                "decision_rule": rule,
                "decision_rule_order": DECISION_RULE_ORDER.index(rule),
                "decision_cutoff_hkt": cutoff_hkt,
                "decision_cutoff_utc": cutoff_hkt.tz_convert("UTC"),
                "decision_cutoff_unix": int(cutoff_hkt.timestamp()),
            }
        )
    return rows

In [3]:
outcomes = pd.read_csv(
    OUTCOME_PATH,
    dtype={
        "market_id": str,
        "condition_id": str,
        "yes_token_id": str,
        "no_token_id": str,
    },
)
outcomes["event_date"] = pd.to_datetime(outcomes["event_date"])

required_columns = {
    "event_date",
    "event_id",
    "event_slug",
    "market_id",
    "condition_id",
    "market_slug",
    "question",
    "canonical_label",
    "event_type",
    "yes_token_id",
    "no_token_id",
    "hko_daily_max_c",
    "realised_yes",
    "realised_no",
}
missing_columns = required_columns.difference(outcomes.columns)
if missing_columns:
    raise AssertionError(
        "18o outcome panel is missing columns: "
        f"{sorted(missing_columns)}"
    )

if len(outcomes) != 330:
    raise AssertionError(
        f"Expected 330 June outcomes, found {len(outcomes)}"
    )
if outcomes["event_date"].nunique() != 30:
    raise AssertionError(
        "Expected 30 unique June dates"
    )
if outcomes.duplicated(["event_date", "market_id"]).any():
    raise AssertionError(
        "Duplicate date-market keys in 18o input"
    )
if outcomes["yes_token_id"].isna().any():
    raise AssertionError(
        "At least one YES token is missing"
    )
if outcomes["yes_token_id"].nunique() != 330:
    raise AssertionError(
        "YES token identifiers are not unique"
    )
if not outcomes["realised_yes"].isin([0, 1]).all():
    raise AssertionError(
        "Non-binary realised outcomes found"
    )
if not outcomes.groupby("event_date")["realised_yes"].sum().eq(1).all():
    raise AssertionError(
        "At least one event book does not have exactly one winner"
    )

event_type_map = {
    "lower": "lower_tail_endpoint",
    "interior": "interior_bin",
    "upper": "upper_tail",
}
outcomes["contract_event_type_v2"] = outcomes["event_type"].map(event_type_map)

if outcomes["contract_event_type_v2"].isna().any():
    raise AssertionError("Unexpected contract event type")

token_inventory = (
    outcomes[
        [
            "event_date",
            "event_id",
            "event_slug",
            "market_id",
            "condition_id",
            "market_slug",
            "question",
            "canonical_label",
            "event_type",
            "contract_event_type_v2",
            "yes_token_id",
            "no_token_id",
            "hko_daily_max_c",
            "realised_yes",
            "realised_no",
        ]
    ]
    .rename(columns={"yes_token_id": "selected_yes_token_id"})
    .sort_values(["event_date", "market_id"])
    .reset_index(drop=True)
)

print(f"Input contracts: {len(token_inventory)}")
print(f"Input dates: {token_inventory['event_date'].nunique()}")
print(f"Unique YES tokens: {token_inventory['selected_yes_token_id'].nunique()}")

Input contracts: 330
Input dates: 30
Unique YES tokens: 330


In [4]:
fetch_rows: list[dict[str, Any]] = []
history_rows: list[dict[str, Any]] = []

for row_number, row in enumerate(token_inventory.itertuples(index=False), start=1):
    event_date = pd.Timestamp(row.event_date)
    event_open_hkt = pd.Timestamp(
        year=event_date.year,
        month=event_date.month,
        day=event_date.day,
        hour=0,
        minute=0,
        second=0,
        tz=HKT,
    )
    start_hkt = event_open_hkt - pd.Timedelta(days=HISTORY_WINDOW_DAYS)
    end_hkt = event_open_hkt

    start_ts = int(start_hkt.timestamp())
    end_ts = int(end_hkt.timestamp())
    token_id = str(row.selected_yes_token_id)

    raw_path = (
        RAW_DIR
        / f"{event_date.date().isoformat()}__{row.market_id}__{token_id}.json"
    )

    payload, metadata = request_price_history(
        token_id=token_id,
        start_ts=start_ts,
        end_ts=end_ts,
    )

    raw_path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    metadata.update(
        {
            "event_date": event_date.date().isoformat(),
            "market_id": str(row.market_id),
            "market_slug": row.market_slug,
            "group_item_title": row.canonical_label,
            "attempt_label": (
                f"window_{HISTORY_WINDOW_DAYS}d_"
                f"fidelity_{HISTORY_FIDELITY_MINUTES}"
            ),
            "fetch_source": "api",
            "raw_response_path": str(raw_path.relative_to(REPO_ROOT)),
            "raw_response_size_bytes": raw_path.stat().st_size,
            "raw_response_sha256": sha256_file(raw_path),
        }
    )
    fetch_rows.append(metadata)

    history_rows.extend(
        normalise_history_points(
            payload=payload,
            token_id=token_id,
            event_date=event_date,
            raw_path=raw_path,
        )
    )

    if row_number % 25 == 0 or row_number == len(token_inventory):
        print(
            f"Fetched {row_number}/{len(token_inventory)} tokens; "
            f"history rows accumulated={len(history_rows)}"
        )

    time.sleep(REQUEST_SLEEP_SECONDS)

fetch_inventory = pd.DataFrame(fetch_rows)
price_history = pd.DataFrame(
    history_rows,
    columns=[
        "event_date",
        "selected_yes_token_id",
        "price_timestamp_unix",
        "price_timestamp_utc",
        "price_timestamp_hkt",
        "p_market",
        "source_index",
        "raw_response_path",
    ],
)

if not price_history.empty:
    price_history["event_date"] = pd.to_datetime(price_history["event_date"])
    price_history["price_timestamp_utc"] = pd.to_datetime(
        price_history["price_timestamp_utc"], utc=True
    )
    price_history["price_timestamp_hkt"] = pd.to_datetime(
        price_history["price_timestamp_hkt"], utc=True
    ).dt.tz_convert(HKT)

    price_history = (
        price_history.sort_values(
            [
                "selected_yes_token_id",
                "price_timestamp_unix",
                "source_index",
            ]
        )
        .drop_duplicates(
            ["selected_yes_token_id", "price_timestamp_unix"],
            keep="last",
        )
        .reset_index(drop=True)
    )

print(f"Tokens with history: {int(fetch_inventory['has_history'].sum())}")
print(f"Tokens without history: {int((~fetch_inventory['has_history']).sum())}")
print(f"Recovered price-history rows: {len(price_history)}")

Fetched 25/330 tokens; history rows accumulated=875


Fetched 50/330 tokens; history rows accumulated=1750


Fetched 75/330 tokens; history rows accumulated=2136


Fetched 100/330 tokens; history rows accumulated=2796


Fetched 125/330 tokens; history rows accumulated=3671


...

Fetched 275/330 tokens; history rows accumulated=8864


Fetched 300/330 tokens; history rows accumulated=9739


Fetched 325/330 tokens; history rows accumulated=10614


Fetched 330/330 tokens; history rows accumulated=10789


Tokens with history: 330
Tokens without history: 0
Recovered price-history rows: 10789


In [5]:
decision_grid_rows: list[dict[str, Any]] = []

for row in token_inventory.itertuples(index=False):
    for decision in decision_times(pd.Timestamp(row.event_date)):
        decision_grid_rows.append(
            {
                "event_date": pd.Timestamp(row.event_date),
                "event_id": row.event_id,
                "event_slug": row.event_slug,
                "market_id": row.market_id,
                "condition_id": row.condition_id,
                "market_slug": row.market_slug,
                "question": row.question,
                "group_item_title": row.canonical_label,
                "event_type": row.event_type,
                "contract_event_type_v2": row.contract_event_type_v2,
                "selected_yes_token_id": row.selected_yes_token_id,
                "no_token_id": row.no_token_id,
                "hko_daily_max_c": row.hko_daily_max_c,
                "Y_event_int": int(row.realised_yes),
                "Y_no_int": int(row.realised_no),
                **decision,
            }
        )

decision_grid = pd.DataFrame(decision_grid_rows)

if len(decision_grid) != 1320:
    raise AssertionError(
        f"Expected 1,320 decision candidates, found {len(decision_grid)}"
    )
if decision_grid.duplicated(
    ["event_date", "market_id", "decision_rule"]
).any():
    raise AssertionError(
        "Duplicate date-market-rule decision keys"
    )

history_by_token = {
    token: group.sort_values(
        ["price_timestamp_unix", "source_index"]
    ).reset_index(drop=True)
    for token, group in price_history.groupby("selected_yes_token_id")
}

selected_rows: list[dict[str, Any]] = []
issue_rows: list[dict[str, Any]] = []

for row in decision_grid.itertuples(index=False):
    candidate_history = history_by_token.get(
        str(row.selected_yes_token_id)
    )
    selected = None

    if candidate_history is not None and not candidate_history.empty:
        eligible = candidate_history.loc[
            candidate_history["price_timestamp_unix"]
            <= int(row.decision_cutoff_unix)
        ]
        if not eligible.empty:
            selected = eligible.iloc[-1]

    base = row._asdict()

    if selected is None:
        base.update(
            {
                "price_available": False,
                "p_market": np.nan,
                "selected_price_timestamp_unix": pd.NA,
                "selected_price_timestamp_utc": pd.NaT,
                "selected_price_timestamp_hkt": pd.NaT,
                "price_staleness_hours": np.nan,
                "selected_price_raw_response_path": "",
            }
        )
        issue_rows.append(
            {
                "issue_type": "missing_decision_price",
                "event_date": row.event_date.date().isoformat(),
                "market_id": row.market_id,
                "market_slug": row.market_slug,
                "group_item_title": row.group_item_title,
                "selected_yes_token_id": row.selected_yes_token_id,
                "decision_rule": row.decision_rule,
                "decision_cutoff_utc": row.decision_cutoff_utc,
                "detail": (
                    "No price point at or before the decision cutoff "
                    "within the recovered seven-day history window."
                ),
            }
        )
    else:
        price_timestamp_utc = pd.Timestamp(selected["price_timestamp_utc"])
        staleness_hours = (
            pd.Timestamp(row.decision_cutoff_utc) - price_timestamp_utc
        ).total_seconds() / 3600.0

        base.update(
            {
                "price_available": True,
                "p_market": float(selected["p_market"]),
                "selected_price_timestamp_unix": int(
                    selected["price_timestamp_unix"]
                ),
                "selected_price_timestamp_utc": price_timestamp_utc,
                "selected_price_timestamp_hkt": pd.Timestamp(
                    selected["price_timestamp_hkt"]
                ),
                "price_staleness_hours": staleness_hours,
                "selected_price_raw_response_path": selected[
                    "raw_response_path"
                ],
            }
        )

    selected_rows.append(base)

decision_panel = pd.DataFrame(selected_rows)
missing_prices = pd.DataFrame(
    issue_rows,
    columns=[
        "issue_type",
        "event_date",
        "market_id",
        "market_slug",
        "group_item_title",
        "selected_yes_token_id",
        "decision_rule",
        "decision_cutoff_utc",
        "detail",
    ],
)

scoring_panel = decision_panel.loc[
    decision_panel["price_available"]
].copy()

clipped = scoring_panel["p_market"].clip(
    LOG_EPSILON,
    1.0 - LOG_EPSILON,
)
scoring_panel["brier_market"] = (
    scoring_panel["p_market"] - scoring_panel["Y_event_int"]
) ** 2
scoring_panel["log_score_market"] = -(
    scoring_panel["Y_event_int"] * np.log(clipped)
    + (1 - scoring_panel["Y_event_int"]) * np.log(1 - clipped)
)

print(f"Decision candidates: {len(decision_panel)}")
print(f"Decision prices recovered: {len(scoring_panel)}")
print(f"Missing decision prices: {len(missing_prices)}")

Decision candidates: 1320
Decision prices recovered: 1265
Missing decision prices: 55


In [6]:
# Binary score summaries: ALL plus event-type breakdown.
score_summary_rows: list[dict[str, Any]] = []

for decision_rule in DECISION_RULE_ORDER:
    rule_frame = scoring_panel.loc[
        scoring_panel["decision_rule"].eq(decision_rule)
    ]

    for event_type_label, subset in [
        ("ALL", rule_frame),
        *[
            (
                event_type,
                rule_frame.loc[
                    rule_frame["contract_event_type_v2"].eq(event_type)
                ],
            )
            for event_type in [
                "interior_bin",
                "lower_tail_endpoint",
                "upper_tail",
            ]
        ],
    ]:
        if subset.empty:
            continue

        score_summary_rows.append(
            {
                "decision_rule": decision_rule,
                "decision_rule_order": DECISION_RULE_ORDER.index(
                    decision_rule
                ),
                "contract_event_type_v2": event_type_label,
                "n": len(subset),
                "mean_brier": subset["brier_market"].mean(),
                "median_brier": subset["brier_market"].median(),
                "mean_log_score": subset["log_score_market"].mean(),
                "median_log_score": subset["log_score_market"].median(),
                "mean_p_market": subset["p_market"].mean(),
                "outcome_rate": subset["Y_event_int"].mean(),
                "median_staleness_hours": subset[
                    "price_staleness_hours"
                ].median(),
                "p95_staleness_hours": subset[
                    "price_staleness_hours"
                ].quantile(0.95),
            }
        )

market_score_summary = pd.DataFrame(score_summary_rows).sort_values(
    ["decision_rule_order", "contract_event_type_v2"]
)

# Event-book diagnostics.
book_rows: list[dict[str, Any]] = []

for (event_date, decision_rule), book in scoring_panel.groupby(
    ["event_date", "decision_rule"],
    sort=True,
):
    book = book.copy()
    n_snapshots = len(book)
    total_probability = float(book["p_market"].sum())
    n_yes = int(book["Y_event_int"].sum())
    full_book = n_snapshots == 11 and n_yes == 1

    winner_rows = book.loc[book["Y_event_int"].eq(1)]
    winning_probability_raw = (
        float(winner_rows["p_market"].iloc[0])
        if len(winner_rows) == 1
        else np.nan
    )
    winning_probability_normalised = (
        winning_probability_raw / total_probability
        if full_book and total_probability > 0
        else np.nan
    )

    raw_categorical_log_score = (
        -math.log(
            min(
                max(winning_probability_raw, LOG_EPSILON),
                1.0 - LOG_EPSILON,
            )
        )
        if full_book
        else np.nan
    )
    normalised_categorical_log_score = (
        -math.log(
            min(
                max(winning_probability_normalised, LOG_EPSILON),
                1.0 - LOG_EPSILON,
            )
        )
        if full_book
        else np.nan
    )

    raw_multiclass_brier = (
        float(
            np.square(
                book["p_market"].to_numpy()
                - book["Y_event_int"].to_numpy()
            ).sum()
        )
        if full_book
        else np.nan
    )

    if full_book and total_probability > 0:
        normalised_probabilities = (
            book["p_market"].to_numpy() / total_probability
        )
        normalised_multiclass_brier = float(
            np.square(
                normalised_probabilities
                - book["Y_event_int"].to_numpy()
            ).sum()
        )
    else:
        normalised_multiclass_brier = np.nan

    book_rows.append(
        {
            "event_date": event_date,
            "decision_rule": decision_rule,
            "decision_rule_order": DECISION_RULE_ORDER.index(decision_rule),
            "n_price_snapshots": n_snapshots,
            "n_yes_contracts": n_yes,
            "full_book": full_book,
            "total_book_market_probability": total_probability,
            "winning_contract_market_probability": winning_probability_raw,
            "winning_contract_market_probability_normalised": (
                winning_probability_normalised
            ),
            "book_probability_error_vs_one": total_probability - 1.0,
            "max_price_staleness_hours": book[
                "price_staleness_hours"
            ].max(),
            "median_price_staleness_hours": book[
                "price_staleness_hours"
            ].median(),
            "raw_categorical_log_score": raw_categorical_log_score,
            "normalised_categorical_log_score": (
                normalised_categorical_log_score
            ),
            "raw_multiclass_brier": raw_multiclass_brier,
            "normalised_multiclass_brier": normalised_multiclass_brier,
        }
    )

event_book_summary = pd.DataFrame(book_rows).sort_values(
    ["event_date", "decision_rule_order"]
)

categorical_summary_rows: list[dict[str, Any]] = []
for decision_rule in DECISION_RULE_ORDER:
    subset = event_book_summary.loc[
        event_book_summary["decision_rule"].eq(decision_rule)
    ]
    full = subset.loc[subset["full_book"]]

    categorical_summary_rows.append(
        {
            "decision_rule": decision_rule,
            "decision_rule_order": DECISION_RULE_ORDER.index(decision_rule),
            "n_book_snapshots": len(subset),
            "n_full_books": len(full),
            "full_book_rate": (
                len(full) / len(subset) if len(subset) > 0 else np.nan
            ),
            "mean_total_book_probability": subset[
                "total_book_market_probability"
            ].mean(),
            "median_total_book_probability": subset[
                "total_book_market_probability"
            ].median(),
            "mean_abs_book_probability_error": subset[
                "book_probability_error_vs_one"
            ].abs().mean(),
            "median_winning_probability_raw": full[
                "winning_contract_market_probability"
            ].median(),
            "median_winning_probability_normalised": full[
                "winning_contract_market_probability_normalised"
            ].median(),
            "mean_raw_categorical_log_score": full[
                "raw_categorical_log_score"
            ].mean(),
            "mean_normalised_categorical_log_score": full[
                "normalised_categorical_log_score"
            ].mean(),
            "mean_raw_multiclass_brier": full[
                "raw_multiclass_brier"
            ].mean(),
            "mean_normalised_multiclass_brier": full[
                "normalised_multiclass_brier"
            ].mean(),
            "median_max_staleness_hours": full[
                "max_price_staleness_hours"
            ].median(),
        }
    )

categorical_score_summary = pd.DataFrame(categorical_summary_rows)

display(
    market_score_summary.loc[
        market_score_summary["contract_event_type_v2"].eq("ALL")
    ]
)
display(categorical_score_summary)

,decision_rule,decision_rule_order,contract_event_type_v2,n,mean_brier,median_brier,mean_log_score,median_log_score,mean_p_market,outcome_rate,median_staleness_hours,p95_staleness_hours
0,24h_prior,0,ALL,297,0.058715,0.000380,0.184025,0.019693,0.094088,0.090909,0.998056,0.999167
4,12h_prior,1,ALL,308,0.058737,0.000298,0.184717,0.017401,0.093131,0.090909,0.998333,0.999167
8,6h_prior,2,ALL,330,0.056012,0.000169,0.177046,0.013085,0.093521,0.090909,0.998333,0.999167
12,event_day_open,3,ALL,330,0.055751,0.000086,0.177376,0.009293,0.094024,0.090909,0.998333,0.999167


,decision_rule,decision_rule_order,n_book_snapshots,n_full_books,full_book_rate,mean_total_book_probability,median_total_book_probability,mean_abs_book_probability_error,median_winning_probability_raw,median_winning_probability_normalised,mean_raw_categorical_log_score,mean_normalised_categorical_log_score,mean_raw_multiclass_brier,mean_normalised_multiclass_brier,median_max_staleness_hours
0,24h_prior,0,27,27,1.0,1.034963,1.03750,0.043333,0.3200,0.315294,1.195824,1.229658,0.645870,0.648074,0.998889
1,12h_prior,1,28,28,1.0,1.024446,1.03625,0.038518,0.3275,0.316338,1.215376,1.239007,0.646106,0.648440,0.999028
2,6h_prior,2,30,30,1.0,1.028733,1.03725,0.038533,0.3575,0.341823,1.154793,1.182630,0.616129,0.617790,0.998889
3,event_day_open,3,30,30,1.0,1.034267,1.04875,0.049000,0.3800,0.367948,1.156844,1.189793,0.613258,0.613239,0.998889


In [7]:
integrity_rows: list[dict[str, Any]] = []

def add_check(name: str, passed: bool, detail: str) -> None:
    integrity_rows.append(
        {
            "check": name,
            "passed": bool(passed),
            "detail": detail,
        }
    )

add_check(
    "target_panel_nonempty",
    not token_inventory.empty,
    f"target rows={len(token_inventory)}",
)
add_check(
    "target_rows_have_binary_payoff",
    outcomes["realised_yes"].isin([0, 1]).all(),
    f"bad rows={int((~outcomes['realised_yes'].isin([0, 1])).sum())}",
)
add_check(
    "target_rows_have_yes_tokens",
    outcomes["yes_token_id"].notna().all(),
    f"missing tokens={int(outcomes['yes_token_id'].isna().sum())}",
)
add_check(
    "fetch_inventory_complete",
    len(fetch_inventory) == 330,
    f"fetch rows={len(fetch_inventory)}",
)
add_check(
    "all_fetches_http_200",
    fetch_inventory["status_code"].eq(200).all(),
    f"non-200 rows={int((~fetch_inventory['status_code'].eq(200)).sum())}",
)
add_check(
    "price_history_nonempty",
    not price_history.empty,
    f"price rows={len(price_history)}",
)
add_check(
    "decision_panel_complete_grid",
    len(decision_panel) == 1320,
    f"decision candidate rows={len(decision_panel)}",
)
add_check(
    "decision_panel_unique_keys",
    not decision_panel.duplicated(
        ["event_date", "market_id", "decision_rule"]
    ).any(),
    (
        "duplicate rows="
        f"{int(decision_panel.duplicated(['event_date', 'market_id', 'decision_rule']).sum())}"
    ),
)
add_check(
    "scoring_panel_nonempty",
    not scoring_panel.empty,
    f"scoring rows={len(scoring_panel)}",
)

no_lookahead_violations = scoring_panel.loc[
    pd.to_datetime(
        scoring_panel["selected_price_timestamp_utc"], utc=True
    )
    > pd.to_datetime(scoring_panel["decision_cutoff_utc"], utc=True)
]
add_check(
    "no_lookahead_decision_timestamps",
    no_lookahead_violations.empty,
    f"violations={len(no_lookahead_violations)}",
)

bad_probabilities = scoring_panel.loc[
    ~scoring_panel["p_market"].between(0.0, 1.0)
]
add_check(
    "probabilities_in_unit_interval",
    bad_probabilities.empty,
    f"bad probabilities={len(bad_probabilities)}",
)

negative_staleness = scoring_panel.loc[
    scoring_panel["price_staleness_hours"] < -1e-12
]
add_check(
    "non_negative_staleness",
    negative_staleness.empty,
    f"negative staleness={len(negative_staleness)}",
)
add_check(
    "missing_price_issue_table_written",
    len(missing_prices)
    == int((~decision_panel["price_available"]).sum()),
    f"issue rows={len(missing_prices)}",
)
add_check(
    "some_event_book_snapshots_observed",
    not event_book_summary.empty,
    f"book snapshots={len(event_book_summary)}",
)
add_check(
    "some_full_books_observed",
    bool(event_book_summary["full_book"].any()),
    f"full books={int(event_book_summary['full_book'].sum())}",
)
add_check(
    "full_books_have_one_winner",
    event_book_summary.loc[
        event_book_summary["full_book"],
        "n_yes_contracts",
    ].eq(1).all(),
    (
        "bad full books="
        f"{int((~event_book_summary.loc[event_book_summary['full_book'], 'n_yes_contracts'].eq(1)).sum())}"
    ),
)

integrity_checks = pd.DataFrame(integrity_rows)

if not integrity_checks["passed"].all():
    failed = integrity_checks.loc[~integrity_checks["passed"]]
    raise AssertionError(
        "Integrity checks failed:\n" + failed.to_string(index=False)
    )

verdict = (
    "PASS"
    if len(scoring_panel) > 0 and integrity_checks["passed"].all()
    else "NEEDS_CORRECTION"
)

# Convert time fields to stable strings before writing.
for frame in [price_history, decision_panel, scoring_panel, event_book_summary]:
    if "event_date" in frame.columns:
        frame["event_date"] = pd.to_datetime(frame["event_date"]).dt.date.astype(str)

time_columns_by_frame = {
    id(price_history): [
        "price_timestamp_utc",
        "price_timestamp_hkt",
    ],
    id(decision_panel): [
        "decision_cutoff_hkt",
        "decision_cutoff_utc",
        "selected_price_timestamp_utc",
        "selected_price_timestamp_hkt",
    ],
    id(scoring_panel): [
        "decision_cutoff_hkt",
        "decision_cutoff_utc",
        "selected_price_timestamp_utc",
        "selected_price_timestamp_hkt",
    ],
}

for frame in [price_history, decision_panel, scoring_panel]:
    for column in time_columns_by_frame.get(id(frame), []):
        if column in frame.columns:
            frame[column] = frame[column].astype(str)

output_paths = {
    "price_history": (
        OUT_DIR / "18p_june_2026_clob_yes_price_history.csv"
    ),
    "fetch_inventory": (
        OUT_DIR / "18p_june_2026_clob_fetch_inventory.csv"
    ),
    "decision_panel": (
        OUT_DIR / "18p_june_2026_no_lookahead_decision_panel.csv"
    ),
    "scoring_panel": (
        OUT_DIR / "18p_june_2026_market_scoring_panel.csv"
    ),
    "missing_prices": (
        OUT_DIR / "18p_june_2026_missing_decision_prices.csv"
    ),
    "market_score_summary": (
        OUT_DIR / "18p_june_2026_market_score_summary.csv"
    ),
    "event_book_summary": (
        OUT_DIR / "18p_june_2026_event_book_snapshot_summary.csv"
    ),
    "categorical_score_summary": (
        OUT_DIR / "18p_june_2026_categorical_score_summary.csv"
    ),
    "integrity_checks": (
        OUT_DIR / "18p_june_2026_integrity_checks.csv"
    ),
}

price_history.to_csv(output_paths["price_history"], index=False)
fetch_inventory.to_csv(output_paths["fetch_inventory"], index=False)
decision_panel.to_csv(output_paths["decision_panel"], index=False)
scoring_panel.to_csv(output_paths["scoring_panel"], index=False)
missing_prices.to_csv(output_paths["missing_prices"], index=False)
market_score_summary.to_csv(
    output_paths["market_score_summary"], index=False
)
event_book_summary.to_csv(
    output_paths["event_book_summary"], index=False
)
categorical_score_summary.to_csv(
    output_paths["categorical_score_summary"], index=False
)
integrity_checks.to_csv(
    output_paths["integrity_checks"], index=False
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": verdict,
    "input_contract_rows": int(len(token_inventory)),
    "input_unique_dates": int(token_inventory["event_date"].nunique()),
    "input_unique_yes_tokens": int(
        token_inventory["selected_yes_token_id"].nunique()
    ),
    "history_window_days": HISTORY_WINDOW_DAYS,
    "history_fidelity_minutes": HISTORY_FIDELITY_MINUTES,
    "fetch_rows": int(len(fetch_inventory)),
    "fetch_http_200_rows": int(
        fetch_inventory["status_code"].eq(200).sum()
    ),
    "tokens_with_history": int(fetch_inventory["has_history"].sum()),
    "tokens_without_history": int(
        (~fetch_inventory["has_history"]).sum()
    ),
    "price_history_rows": int(len(price_history)),
    "decision_candidate_rows": int(len(decision_panel)),
    "decision_price_rows": int(len(scoring_panel)),
    "missing_decision_price_rows": int(len(missing_prices)),
    "event_book_snapshots_with_any_prices": int(len(event_book_summary)),
    "full_event_book_snapshots": int(
        event_book_summary["full_book"].sum()
    ),
    "integrity_checks_passed": int(
        integrity_checks["passed"].sum()
    ),
    "integrity_checks_total": int(len(integrity_checks)),
    "methodological_note": (
        "Latest recovered YES-token price at or before each cutoff; "
        "no forward fill and no use of post-cutoff observations."
    ),
}

summary_path = OUT_DIR / "18p_june_2026_market_recovery_summary.json"
summary_path.write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "requests": requests.__version__,
    "clob_price_history_url": PRICE_HISTORY_URL,
    "history_window_days": HISTORY_WINDOW_DAYS,
    "history_fidelity_minutes": HISTORY_FIDELITY_MINUTES,
    "decision_rules": DECISION_RULE_OFFSETS_HOURS,
    "log_epsilon": LOG_EPSILON,
}
environment_path = OUT_DIR / "18p_june_2026_environment.json"
environment_path.write_text(
    json.dumps(environment, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, ensure_ascii=False))
display(integrity_checks)

{
  "step": "18p",
  "generated_at_utc": "2026-07-21T04:37:44.016524+00:00",
  "verdict": "PASS",
  "input_contract_rows": 330,
  "input_unique_dates": 30,
  "input_unique_yes_tokens": 330,
  "history_window_days": 7,
  "history_fidelity_minutes": 60,
  "fetch_rows": 330,
  "fetch_http_200_rows": 330,
  "tokens_with_history": 330,
  "tokens_without_history": 0,
  "price_history_rows": 10789,
  "decision_candidate_rows": 1320,
  "decision_price_rows": 1265,
  "missing_decision_price_rows": 55,
  "event_book_snapshots_with_any_prices": 115,
  "full_event_book_snapshots": 115,
  "integrity_checks_passed": 16,
  "integrity_checks_total": 16,
  "methodological_note": "Latest recovered YES-token price at or before each cutoff; no forward fill and no use of post-cutoff observations."
}


,check,passed,detail
0,target_panel_nonempty,True,target rows=330
1,target_rows_have_binary_payoff,True,bad rows=0
2,target_rows_have_yes_tokens,True,missing tokens=0
3,fetch_inventory_complete,True,fetch rows=330
4,all_fetches_http_200,True,non-200 rows=0
5,price_history_nonempty,True,price rows=10789
6,decision_panel_complete_grid,True,decision candidate rows=1320
7,decision_panel_unique_keys,True,duplicate rows=0
8,scoring_panel_nonempty,True,scoring rows=1265
9,no_lookahead_decision_timestamps,True,violations=0


In [8]:
report_lines = [
    "# 18p June 2026 CLOB market-price recovery",
    "",
    f"Generated at UTC: `{summary['generated_at_utc']}`",
    "",
    "## Overall judgement",
    "",
    f"**{summary['verdict']}**",
    "",
    "## Purpose",
    "",
    (
        "Recover historical Polymarket CLOB YES-token prices for all "
        "330 certified June HKO event contracts and construct strict "
        "no-look-ahead snapshots at the four established decision rules."
    ),
    "",
    "## Main result",
    "",
    f"- Input contract rows: {summary['input_contract_rows']}",
    f"- Unique event dates: {summary['input_unique_dates']}",
    f"- Unique YES tokens: {summary['input_unique_yes_tokens']}",
    f"- Price-history observation rows: {summary['price_history_rows']}",
    f"- Decision candidates: {summary['decision_candidate_rows']}",
    f"- No-look-ahead decision prices: {summary['decision_price_rows']}",
    (
        "- Missing decision-price rows: "
        f"{summary['missing_decision_price_rows']}"
    ),
    (
        "- Event-book snapshots with at least one price: "
        f"{summary['event_book_snapshots_with_any_prices']}"
    ),
    (
        "- Complete eleven-contract event-book snapshots: "
        f"{summary['full_event_book_snapshots']}"
    ),
    "",
    "## CLOB recovery",
    "",
    (
        f"- History window: {HISTORY_WINDOW_DAYS} days ending at "
        "event-day open."
    ),
    f"- Fidelity: {HISTORY_FIDELITY_MINUTES} minutes.",
    (
        "- Selection rule: latest recovered price timestamp at or "
        "before the decision cut-off."
    ),
    "- Missing prices are retained as missing and are not imputed.",
    "",
    "## Decision cut-offs",
    "",
    "| Rule | Hong Kong local time |",
    "|---|---|",
    "| `24h_prior` | 00:00 HKT on the day before the event |",
    "| `12h_prior` | 12:00 HKT on the day before the event |",
    "| `6h_prior` | 18:00 HKT on the day before the event |",
    "| `event_day_open` | 00:00 HKT on the event date |",
    "",
    "## Market-only binary score summary",
    "",
    (
        "| Rule | n | Mean Brier | Mean log score | "
        "Median staleness hours | Mean market probability | Outcome rate |"
    ),
    "|---|---:|---:|---:|---:|---:|---:|",
]

all_scores = market_score_summary.loc[
    market_score_summary["contract_event_type_v2"].eq("ALL")
].sort_values("decision_rule_order")

for row in all_scores.itertuples(index=False):
    report_lines.append(
        "| {rule} | {n} | {brier:.8f} | {log:.8f} | "
        "{stale:.6f} | {p:.8f} | {rate:.8f} |".format(
            rule=row.decision_rule,
            n=int(row.n),
            brier=float(row.mean_brier),
            log=float(row.mean_log_score),
            stale=float(row.median_staleness_hours),
            p=float(row.mean_p_market),
            rate=float(row.outcome_rate),
        )
    )

report_lines.extend(
    [
        "",
        "## Event-book categorical summary",
        "",
        (
            "| Rule | Books | Full books | Full-book rate | "
            "Mean total probability | Mean normalised categorical log | "
            "Mean normalised multiclass Brier |"
        ),
        "|---|---:|---:|---:|---:|---:|---:|",
    ]
)

for row in categorical_score_summary.sort_values(
    "decision_rule_order"
).itertuples(index=False):
    report_lines.append(
        "| {rule} | {books} | {full} | {rate:.6f} | "
        "{total:.6f} | {log:.6f} | {brier:.6f} |".format(
            rule=row.decision_rule,
            books=int(row.n_book_snapshots),
            full=int(row.n_full_books),
            rate=float(row.full_book_rate),
            total=float(row.mean_total_book_probability),
            log=float(row.mean_normalised_categorical_log_score),
            brier=float(row.mean_normalised_multiclass_brier),
        )
    )

report_lines.extend(
    [
        "",
        "## Integrity checks",
        "",
        "| Check | Passed | Detail |",
        "|---|---|---|",
    ]
)

for row in integrity_checks.itertuples(index=False):
    report_lines.append(
        f"| {row.check} | {row.passed} | {row.detail} |"
    )

report_lines.extend(
    [
        "",
        "## Interpretation",
        "",
        (
            "The June market panel is suitable for downstream use where "
            "a decision price is available. Missing cells remain explicit. "
            "The panel does not use any price observation after its "
            "corresponding decision cut-off."
        ),
    ]
)

report_path = (
    REPORT_DIR / "18p_june_2026_clob_market_price_recovery_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

# Generate the manifest after every canonical output is final.
manifest_rows: list[dict[str, Any]] = []
for root in (RAW_DIR, OUT_DIR, REPORT_DIR):
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18p_june_2026_sha256_manifest.csv":
            continue
        manifest_rows.append(
            {
                "path": str(path.relative_to(REPO_ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = OUT_DIR / "18p_june_2026_sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(manifest_path, index=False)

print(f"Report: {report_path.relative_to(REPO_ROOT)}")
print(f"Manifest entries: {len(manifest_rows)}")

Report: reports/18p_june_2026_clob_market_price_recovery/18p_june_2026_clob_market_price_recovery_report.md
Manifest entries: 342


In [9]:
print("Fetch summary:")
display(
    fetch_inventory.groupby(
        ["status_code", "has_history"],
        dropna=False,
    )
    .size()
    .rename("contracts")
    .reset_index()
)

print("Decision availability by rule:")
display(
    decision_panel.groupby("decision_rule")
    .agg(
        candidates=("market_id", "size"),
        prices_available=("price_available", "sum"),
    )
    .reindex(DECISION_RULE_ORDER)
    .assign(
        missing=lambda frame: (
            frame["candidates"] - frame["prices_available"]
        )
    )
    .reset_index()
)

print("Missing-price issue preview:")
if missing_prices.empty:
    print("No missing decision prices.")
else:
    display(missing_prices.head(30))

print(f"Final verdict: {summary['verdict']}")

Fetch summary:


,status_code,has_history,contracts
0,200,True,330


Decision availability by rule:


,decision_rule,candidates,prices_available,missing
0,24h_prior,330,297,33
1,12h_prior,330,308,22
2,6h_prior,330,330,0
3,event_day_open,330,330,0


Missing-price issue preview:


,issue_type,event_date,market_id,market_slug,group_item_title,selected_yes_token_id,decision_rule,decision_cutoff_utc,detail
0,missing_decision_price,2026-06-06,2440921,highest-temperature-in-hong-kong-on-june-6-202...,28°C or below,6641176110548908385028494983520731616574722814...,24h_prior,2026-06-04 16:00:00+00:00,No price point at or before the decision cutof...
1,missing_decision_price,2026-06-06,2440921,highest-temperature-in-hong-kong-on-june-6-202...,28°C or below,6641176110548908385028494983520731616574722814...,12h_prior,2026-06-05 04:00:00+00:00,No price point at or before the decision cutof...
2,missing_decision_price,2026-06-06,2440922,highest-temperature-in-hong-kong-on-june-6-202...,29°C,1047152563190139604668221635781140205479668597...,24h_prior,2026-06-04 16:00:00+00:00,No price point at or before the decision cutof...
3,missing_decision_price,2026-06-06,2440922,highest-temperature-in-hong-kong-on-june-6-202...,29°C,1047152563190139604668221635781140205479668597...,12h_prior,2026-06-05 04:00:00+00:00,No price point at or before the decision cutof...
4,missing_decision_price,2026-06-06,2440923,highest-temperature-in-hong-kong-on-june-6-202...,30°C,1023940479297540631228349934198473568811256463...,24h_prior,2026-06-04 16:00:00+00:00,No price point at or before the decision cutof...
5,missing_decision_price,2026-06-06,2440923,highest-temperature-in-hong-kong-on-june-6-202...,30°C,1023940479297540631228349934198473568811256463...,12h_prior,2026-06-05 04:00:00+00:00,No price point at or before the decision cutof...
6,missing_decision_price,2026-06-06,2440924,highest-temperature-in-hong-kong-on-june-6-202...,31°C,5972431926294886618906569651057372176238425351...,24h_prior,2026-06-04 16:00:00+00:00,No price point at or before the decision cutof...
7,missing_decision_price,2026-06-06,2440924,highest-temperature-in-hong-kong-on-june-6-202...,31°C,5972431926294886618906569651057372176238425351...,12h_prior,2026-06-05 04:00:00+00:00,No price point at or before the decision cutof...
8,missing_decision_price,2026-06-06,2440925,highest-temperature-in-hong-kong-on-june-6-202...,32°C,5307091267889838378934579427711535130506393308...,24h_prior,2026-06-04 16:00:00+00:00,No price point at or before the decision cutof...
9,missing_decision_price,2026-06-06,2440925,highest-temperature-in-hong-kong-on-june-6-202...,32°C,5307091267889838378934579427711535130506393308...,12h_prior,2026-06-05 04:00:00+00:00,No price point at or before the decision cutof...


Final verdict: PASS
